# 01 — Backpropagation: From a Single Neuron to a Full Layer

## What this notebook covers

This is the foundation of everything in deep learning. Before any fancy optimiser or regulariser, we need to understand **how gradients flow backwards** through a network.

We'll build intuition step-by-step:
1. Gradient descent on a single neuron
2. Backprop through one layer with multiple neurons

---

## The core idea: the Chain Rule

When we compose functions — like `loss(relu(linear(x)))` — the gradient of the loss with respect to any parameter is the **product of all the local derivatives** along the path:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w}$$

Where:
- $z = w \cdot x + b$ is the linear combination
- $\hat{y} = \text{ReLU}(z)$ is the activation
- $L = (\hat{y} - y_{\text{true}})^2$ is the squared loss


## Part 1 — Single Neuron

One neuron, three inputs, one output, squared error loss.

In [ ]:
import numpy as np

# --- Setup ---
inputs  = np.array([0.8, -2.3, 5.9])
weights = np.array([-2.0, -5.0, 0.0])
bias    = 0.8
y_true  = 0.0
lr      = 0.0001

def relu(x):             return np.maximum(0, x)
def relu_grad(x):        return np.where(x > 0, 1.0, 0.0)
def mse_loss(pred, true): return (pred - true) ** 2

losses = []

for i in range(400):
    # Forward pass
    z    = np.dot(inputs, weights) + bias
    yhat = relu(z)
    loss = mse_loss(yhat, y_true)
    losses.append(loss)

    # Backward pass (chain rule)
    dL_dyhat    = 2 * (yhat - y_true)     # dL/d(yhat)
    dyhat_dz    = relu_grad(z)             # d(yhat)/dz
    dz_dweights = inputs                   # dz/dw
    dz_dbias    = 1.0                      # dz/db

    dL_dweights = dL_dyhat * dyhat_dz * dz_dweights
    dL_dbias    = dL_dyhat * dyhat_dz * dz_dbias

    weights -= lr * dL_dweights
    bias    -= lr * dL_dbias

    if i % 80 == 0:
        print(f'step {i:>4d} | loss = {loss:.6f}')


## Plotting the loss curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.title('Single-Neuron Training — Loss over Steps')
plt.xlabel('Step'); plt.ylabel('MSE Loss')
plt.grid(True); plt.tight_layout(); plt.show()


## Part 2 — One Full Layer (3 neurons, 4 inputs)

Now we extend to a vector of neurons. The key difference: gradients become **matrices**.

The weight gradient is an **outer product**:
$$\frac{\partial L}{\partial W} = \delta^T \cdot x$$
where $\delta$ is the gradient flowing from the loss back to each neuron.


In [ ]:
inputs  = np.array([0.5, 5.0, 8.0, 6.9])
weights = np.random.randn(3, 4)
bias    = np.array([0.5, 0.9, 5.6])
lr      = 0.001

losses = []

for step in range(20):
    z    = np.dot(inputs, weights.T) + bias
    a    = relu(z)
    y    = np.sum(a)
    loss = y ** 2
    losses.append(loss)

    # Backward
    dL_dy  = 2 * y
    dy_da  = np.ones_like(a)
    da_dz  = relu_grad(z)
    delta  = dL_dy * dy_da * da_dz

    dL_dW = np.outer(delta, inputs)   # (3,4)
    dL_db = delta                     # (3,)

    weights -= lr * dL_dW
    bias    -= lr * dL_db

    print(f'step {step+1:>2d} | loss = {loss:.4f}')


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, marker='o')
plt.title('One-Layer Training — Loss over Steps')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.grid(True); plt.tight_layout(); plt.show()
